# Training Targeted Adversarial Attack for Live Demo

This notebook demonstrates the process of training a targeted attack using the Carlini-Wagner (CW) method to inject specific text into the transcription output of Whisper.

**Goal**: Train a perturbation that makes Whisper transcribe "This is a Demo - aai590" when the user says a neutral phrase (e.g., "Welcome to my demonstration").

## 1. Setup and Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import whisper
from pathlib import Path

# Add project src to path
sys.path.append('..')

from src.attacks.cw import CarliniWagnerAttack

## 2. Configuration and Hyperparameters

Critical settings based on `AGENTS.md`:
- **Sampling Rate**: 16kHz (Whisper requirement)
- **Epsilon**: 0.03 - 0.05 (Imperceptible perturbation)
- **Iterations**: 1500 - 2500 (CW is expensive)
- **Batch Size**: 1 (To save VRAM during gradient accumulation)

In [ ]:
# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_SIZE = 'small'  # Use small for faster training during demo

TARGET_TRANSCRIPT = "This is a Demo - aai590"
GENERIC_PHRASE = "Welcome to my demonstration"

CW_CONFIG = {
    'epsilon': 0.03,
    'alpha': 0.015,  # Step size for gradient descent
    'iterations': 2000,
    'learning_rate': 0.01,
    'confidence': 1.0,
    'batch_size': 1,
    'targeted': True,
    'device': DEVICE
}

## 3. Load Whisper Model

Load the model. We use the small variant to keep training times manageable for the demo.

In [ ]:
print(f"Loading Whisper model ({MODEL_SIZE}) on {DEVICE}...")
model = whisper.load_model(MODEL_SIZE, device=DEVICE)
model.eval()  # Set to evaluation mode

## 4. Data Loading

Function to load a clean audio file. We assume the user has recorded the `GENERIC_PHRASE`.

In [ ]:
def load_audio_file(file_path, target_sr=16000):
    """Load audio and ensure it matches Whisper's 16kHz requirement."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Audio file not found: {file_path}")
    
    # Whisper loads audio internally, but we need to ensure the specific file 
    # is handled correctly for the attack.
    audio = whisper.load_audio(file_path)
    audio = whisper.pad_or_trim(audio)
    
    # Ensure float32 and correct channel count (1 for mono)
    audio = audio.astype(np.float32)
    
    return audio

## 5. Training the CW Attack

This cell performs the actual adversarial training.

### Key Implementation Details:
1.  **Gradient Propagation**: The input audio tensor must have `requires_grad=True` before being processed by Whisper.
2.  **Differentiable Preprocessing**: We use Whisper's internal Mel-spectrogram logic which is PyTorch-native and differentiable.
3.  **Loss Function**: We use the CTC Loss to minimize the likelihood of the target transcript.

In [ ]:
# Initialize CW Attack
cw_attack = CarliniWagnerAttack(**CW_CONFIG)

# Prepare input
# We assume the file 'demo_assets/my_voice_generic.wav' exists. 
# If not, create a dummy tensor for demonstration of code structure.

input_audio_path = "demo_assets/my_voice_generic.wav"

try:
    clean_audio = load_audio_file(input_audio_path)
except FileNotFoundError:
    print(f"Note: {input_audio_path} not found. Creating dummy 30s audio.")
    clean_audio = torch.randn(1, 48000 * 30) # Dummy 30s mono at 16kHz

# Convert to tensor with grad enabled (CRITICAL)
x_clean = torch.tensor(clean_audio, dtype=torch.float32, requires_grad=True, device=DEVICE)

print(f"Starting targeted attack training...")
print(f"Target Transcript: '{TARGET_TRANSCRIPT}'")
print(f"Epsilon: {CW_CONFIG['epsilon']}")
print(f"Iterations: {CW_CONFIG['iterations']}")

# Run the attack
adv_audio = cw_attack.attack(
    model=model,
    clean_audio=x_clean,
    target_text=TARGET_TRANSCRIPT,
    model_size=MODEL_SIZE
)

print("Attack completed.")

## 6. Verification and Metrics

1.  **SNR Calculation**: Measures how much noise is added.
2.  **Transcript Comparison**: Show the original vs. adversarial output.

In [ ]:
def calculate_snr(clean, adv):
    """Signal-to-Noise Ratio (SNR) in dB."""
    signal_power = torch.mean(clean ** 2)
    noise_power = torch.mean((adv - clean) ** 2)
    if noise_power == 0:
        return float('inf')
    return 10 * torch.log10(signal_power / noise_power)

# Compute SNR
snr_val = calculate_snr(x_clean.cpu().numpy(), adv.cpu().numpy())
print(f"SNR of Attack: {snr_val:.2f} dB")

# Transcribe Original
model.eval()
with torch.no_grad():
    result_clean = model.transcribe(whisper.pad_or_trim(x_clean.cpu().numpy()), fp16=False)
print(f"\nOriginal Transcript: '{result_clean['text'].strip()}'")

# Transcribe Adversarial
with torch.no_grad():
    result_adv = model.transcribe(whisper.pad_or_trim(adv.cpu().numpy()), fp16=False)
print(f"Adversarial Transcript: '{result_adv['text'].strip()}'")

## 7. Save the Perturbation

Save the perturbation vector `v` such that `adv = clean + v`.
In the Live UI, we will use this `v` to generate attacks on-the-fly.

In [ ]:
# The perturbation is adv_audio - clean_audio
perturbation = adv_audio - x_clean

save_path = "demo_assets/targeted_perturbation.pt"
torch.save(perturbation.cpu(), save_path)

print(f"\nTargeted perturbation saved to: {save_path}")
print(f"Shape: {perturbation.shape}")